# Phase 9: Statistical Analysis & Hypothesis Testing
## IBM Data Analyst Professional Certificate Capstone Project

**Author**: Raghavendra Golla  
**Theme**: Rigorous Inferential Statistics & Business Decision Validation  

---

### 1. Executive Context & Objectives
Data analysts frequently confuse exploratory visual differences with true statistical significance. In this notebook, we apply formal **inferential statistics and hypothesis testing** to test 3 critical talent assumptions for our consulting firm:

1. **Education Premium Test**: Do Master's degree holders command significantly higher compensation than Bachelor's degree holders?
2. **Workplace Productivity Test**: Do remote developers work significantly fewer (or more) weekly hours than office-based developers?
3. **Talent Retention Test**: Is job dissatisfaction statistically linked to active job-seeking behavior (Chi-Square Test of Independence)?

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Make the project's src/ package importable so this notebook calls the same
# tested functions as src/analysis.py instead of re-implementing them.
PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.analysis import (
    run_degree_compensation_test,
    run_remote_work_hours_test,
    run_job_satisfaction_chi2_test,
)

# Load cleaned dataset
CLEAN_PATH = os.path.join("..", "data", "processed", "cleaned_survey_data.csv")
if not os.path.exists(CLEAN_PATH):
    CLEAN_PATH = os.path.join("data", "processed", "cleaned_survey_data.csv")

df = pd.read_csv(CLEAN_PATH)
print(f"Loaded {len(df):,} cleaned records for statistical evaluation.")

Loaded 11,398 cleaned records for statistical evaluation.


---
### 2. Hypothesis Test 1: Compensation Differences by Degree Level
- **Business Question**: Should our consulting firm offer a formal salary premium or mandate Master's degrees for technical hires?
- **H0 (Null Hypothesis)**: The distribution of compensation for Bachelor's and Master's degree holders is identical.
- **H1 (Alternative Hypothesis)**: The distribution of compensation for Bachelor's and Master's degree holders differs significantly.
- **Significance Level**: $\alpha = 0.05$.
- **Test Selected**: Non-parametric **Mann-Whitney U Test** (justified due to right-skewed non-normal compensation distribution) + Welch's t-test for reference.

In [2]:
result_degree = run_degree_compensation_test(df)

print(f"Bachelor's Sample Size: N = {result_degree['sample_bachelor']:,} | Median = ${result_degree['median_bachelor']:,.2f}")
print(f"Master's Sample Size:   N = {result_degree['sample_master']:,} | Median = ${result_degree['median_master']:,.2f}")
print(f"\nMann-Whitney U Test Statistic: {result_degree['u_statistic']:,.1f}")
print(f"p-value:                       {result_degree['p_value']:.4f}")

if result_degree["statistically_significant"]:
    print("RESULT: Reject the Null Hypothesis (Statistically Significant difference).")
else:
    print("RESULT: Fail to Reject the Null Hypothesis (No statistically significant difference).")

Bachelor's Sample Size: N = 5,485 | Median = $60,492.00
Master's Sample Size:   N = 2,524 | Median = $57,744.00

Mann-Whitney U Test Statistic: 7,097,773.0
p-value:                       0.0676
RESULT: Fail to Reject the Null Hypothesis (No statistically significant difference).


**Business Conclusion (Test 1)**:  
With $p = 0.0676 > 0.05$, there is **no statistically significant difference** in median compensation between Bachelor's and Master's degree holders.  
**Strategic Action**: Leadership should remove strict Master's degree requirements in recruitment pipelines and evaluate candidates based on demonstrated skills, portfolio projects, and technical competency.

---
### 3. Hypothesis Test 2: Remote vs. In-Office Working Hours
- **Business Question**: Does remote work reduce weekly working hours compared to office work?
- **H0**: Mean weekly hours for remote developers equals mean weekly hours for office developers ($\mu_{\text{home}} = \mu_{\text{office}}$).
- **H1**: Mean weekly hours differ significantly ($\mu_{\text{home}} \neq \mu_{\text{office}}$).
- **Significance Level**: $\alpha = 0.05$.
- **Test Selected**: Two-sample independent Welch's t-test (on valid range 10-80 hours).

In [3]:
result_hours = run_remote_work_hours_test(df)

print(f"Remote Developers:  N = {result_hours['sample_home']:,} | Mean = {result_hours['mean_home_hours']:.2f} hrs/wk")
print(f"Office Developers:  N = {result_hours['sample_office']:,} | Mean = {result_hours['mean_office_hours']:.2f} hrs/wk")
print(f"\nWelch's t-statistic: {result_hours['t_statistic']:.4f}")
print(f"p-value:             {result_hours['p_value']:.4f}")

if result_hours["statistically_significant"]:
    print("RESULT: Reject Null Hypothesis.")
else:
    print("RESULT: Fail to Reject Null Hypothesis (No significant difference in hours).")

Remote Developers:  N = 3,444 | Mean = 41.09 hrs/wk
Office Developers:  N = 6,551 | Mean = 40.93 hrs/wk

Welch's t-statistic: 1.0988
p-value:             0.2719
RESULT: Fail to Reject Null Hypothesis (No significant difference in hours).


**Business Conclusion (Test 2)**:  
With $p = 0.2719 > 0.05$, there is **zero statistically significant difference** in work hours between remote and office workers (difference is less than 10 minutes/week). Remote work does not degrade developer commitment.

---
### 4. Hypothesis Test 3: Job Satisfaction vs. Active Job Seeking
- **Business Question**: How strongly does employee dissatisfaction predict active turnover risk?
- **H0**: Job satisfaction and job-seeking status are independent.
- **H1**: Job satisfaction and job-seeking status are dependent.
- **Significance Level**: $\alpha = 0.05$.
- **Test Selected**: Pearson's Chi-Square Test of Independence ($\chi^2$).

In [4]:
result_chi2 = run_job_satisfaction_chi2_test(df)

print("Contingency Table (Job Satisfaction vs. Job Seeking Status):")
display(result_chi2["contingency_table"])

print(f"\nChi-Square (χ²) Statistic: {result_chi2['chi2_statistic']:.2f}")
print(f"Degrees of Freedom:        {result_chi2['degrees_of_freedom']}")
print(f"p-value:                   {result_chi2['p_value']:.4e}")

if result_chi2["statistically_significant"]:
    print("RESULT: Reject Null Hypothesis (Strong statistical association).")

Contingency Table (Job Satisfaction vs. Job Seeking Status):


JobSeek,I am actively looking for a job,I am not interested in new job opportunities,"I’m not actively looking, but I am open to new opportunities"
JobSat,,,
Neither satisfied nor dissatisfied,193,133,959
Slightly dissatisfied,416,139,1268
Slightly satisfied,267,856,2854
Very dissatisfied,283,129,361
Very satisfied,96,1901,1542



Chi-Square (χ²) Statistic: 2692.38
Degrees of Freedom:        8
p-value:                   0.0000e+00
RESULT: Reject Null Hypothesis (Strong statistical association).


---
### 5. Summary of Hypothesis Tests & Interview Presentation

| Hypothesis Test | Statistical Test Used | Metric / p-value | Decision | Business Implication |
| :--- | :--- | :--- | :--- | :--- |
| **Degree Premium** | Mann-Whitney U Test | $U = 7.09M, p = 0.0676$ | Fail to Reject $H_0$ | Master's degrees do not warrant automatic salary premiums over Bachelor's degrees. |
| **Remote Work Hours** | Welch's t-test | $t = 1.10, p = 0.2719$ | Fail to Reject $H_0$ | Remote work hours (41.1) match office hours (40.9); remote work policies are safe to scale. |
| **Satisfaction & Churn** | Chi-Square Test ($\chi^2$) | $\chi^2 = 2692.4, p < 0.0001$ | Reject $H_0$ | Dissatisfaction strongly drives active job hunting; internal retention programs offer high ROI. |

> *"In Phase 9, I avoided the trap of making business claims based purely on visual charts. I conducted three formal hypothesis tests: a Mann-Whitney U test on degree attainment, a Welch's t-test on remote work hours, and a Chi-Square test of independence on job satisfaction and turnover. Notably, the data proved that Master's degrees do not yield a statistically significant pay premium (p=0.068), and remote workers work identical hours to in-office employees (p=0.272). These statistical findings provided defensible, executive-ready evidence for our hiring and workforce strategy."*